# Experiment 2: Housing Price Prediction

**Objective**: Predict housing prices using:
- a. Deep Feed Forward Network
- b. Simple Linear Regression

**Dataset**: California Housing Dataset (scikit-learn)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

## 2. Load and Explore Dataset

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing()
X = housing.data
y = housing.target

# Create DataFrame for exploration
df = pd.DataFrame(X, columns=housing.feature_names)
df['Price'] = y

print("Dataset Shape:", X.shape)
print("\nFeature Names:", housing.feature_names)
print("\nDataset Description:")
print(housing.DESCR[:1000])

In [ ]:
# Display first few rows
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

## 3. Data Preprocessing

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling applied.")
print(f"Scaled Training Data - Mean: {X_train_scaled.mean(axis=0).round(4)}")
print(f"Scaled Training Data - Std: {X_train_scaled.std(axis=0).round(4)}")

## 4. Data Visualization

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Price (in $100,000)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Housing Prices', fontweight='bold')
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Mean: ${y.mean()*100000:,.0f}')
axes[0].legend()

# Correlation heatmap
corr_matrix = df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            fmt='.2f', ax=axes[1], square=True)
axes[1].set_title('Feature Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('../data/housing_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## 5a. Simple Linear Regression

In [ ]:
# Train Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

# Evaluate
lr_metrics = {
    'train_mse': mean_squared_error(y_train, y_pred_lr_train),
    'test_mse': mean_squared_error(y_test, y_pred_lr_test),
    'train_mae': mean_absolute_error(y_train, y_pred_lr_train),
    'test_mae': mean_absolute_error(y_test, y_pred_lr_test),
    'train_r2': r2_score(y_train, y_pred_lr_train),
    'test_r2': r2_score(y_test, y_pred_lr_test)
}

print("Linear Regression Results:")
print("="*40)
print(f"Training MSE: {lr_metrics['train_mse']:.4f}")
print(f"Test MSE: {lr_metrics['test_mse']:.4f}")
print(f"Training MAE: {lr_metrics['train_mae']:.4f}")
print(f"Test MAE: {lr_metrics['test_mae']:.4f}")
print(f"Training R²: {lr_metrics['train_r2']:.4f}")
print(f"Test R²: {lr_metrics['test_r2']:.4f}")

In [ ]:
# Linear Regression coefficients
print("\nFeature Coefficients:")
for name, coef in zip(housing.feature_names, lr_model.coef_):
    print(f"  {name}: {coef:.4f}")
print(f"\nIntercept: {lr_model.intercept_:.4f}")

## 5b. Deep Feed Forward Network

In [ ]:
# Build Deep Feed Forward Network
def build_dnn_model(input_shape):
    model = keras.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(16, activation='relu'),
        layers.Dense(1)  # Output layer for regression
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

# Create model
dnn_model = build_dnn_model(X_train_scaled.shape[1])
dnn_model.summary()

In [ ]:
# Train DNN model
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

history = dnn_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Evaluate DNN model
y_pred_dnn_train = dnn_model.predict(X_train_scaled, verbose=0).flatten()
y_pred_dnn_test = dnn_model.predict(X_test_scaled, verbose=0).flatten()

dnn_metrics = {
    'train_mse': mean_squared_error(y_train, y_pred_dnn_train),
    'test_mse': mean_squared_error(y_test, y_pred_dnn_test),
    'train_mae': mean_absolute_error(y_train, y_pred_dnn_train),
    'test_mae': mean_absolute_error(y_test, y_pred_dnn_test),
    'train_r2': r2_score(y_train, y_pred_dnn_train),
    'test_r2': r2_score(y_test, y_pred_dnn_test)
}

print("\nDeep Feed Forward Network Results:")
print("="*40)
print(f"Training MSE: {dnn_metrics['train_mse']:.4f}")
print(f"Test MSE: {dnn_metrics['test_mse']:.4f}")
print(f"Training MAE: {dnn_metrics['train_mae']:.4f}")
print(f"Test MAE: {dnn_metrics['test_mae']:.4f}")
print(f"Training R²: {dnn_metrics['train_r2']:.4f}")
print(f"Test R²: {dnn_metrics['test_r2']:.4f}")

## 6. Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('DNN Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('DNN Training MAE', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/housing_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Comparison

In [ ]:
# Compare predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear Regression predictions
axes[0].scatter(y_test, y_pred_lr_test, alpha=0.5, edgecolors='k', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].set_title(f'Linear Regression (R² = {lr_metrics["test_r2"]:.4f})', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# DNN predictions
axes[1].scatter(y_test, y_pred_dnn_test, alpha=0.5, edgecolors='k', linewidth=0.5, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].set_title(f'Deep Feed Forward Network (R² = {dnn_metrics["test_r2"]:.4f})', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/housing_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Summary comparison
comparison_df = pd.DataFrame({
    'Metric': ['MSE (Train)', 'MSE (Test)', 'MAE (Train)', 'MAE (Test)', 'R² (Train)', 'R² (Test)'],
    'Linear Regression': [
        lr_metrics['train_mse'], lr_metrics['test_mse'],
        lr_metrics['train_mae'], lr_metrics['test_mae'],
        lr_metrics['train_r2'], lr_metrics['test_r2']
    ],
    'Deep FeedForward Network': [
        dnn_metrics['train_mse'], dnn_metrics['test_mse'],
        dnn_metrics['train_mae'], dnn_metrics['test_mae'],
        dnn_metrics['train_r2'], dnn_metrics['test_r2']
    ]
})

print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(comparison_df.to_string(index=False))

## 8. Conclusion

In [ ]:
print("\n" + "="*60)
print("EXPERIMENT 2 SUMMARY: Housing Price Prediction")
print("="*60)

winner = "Deep Feed Forward Network" if dnn_metrics['test_r2'] > lr_metrics['test_r2'] else "Linear Regression"
improvement = abs(dnn_metrics['test_r2'] - lr_metrics['test_r2']) * 100

print(f"\nBest Model: {winner}")
print(f"R² Improvement: {improvement:.2f}%")

print("\nKey Observations:")
print("- Linear Regression: Simple, interpretable, fast training")
print("- DNN: Can capture non-linear relationships, requires more data")
print("- Both models benefit from feature scaling (StandardScaler)")
print("- DNN uses dropout and batch normalization to prevent overfitting")